In [10]:
import re
import nltk
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [11]:
data = pd.read_csv("Scrapped_sentimental_data.csv")
print(data.sentiment_category.value_counts())

data = data.drop_duplicates(subset="text").reset_index(drop=True)

mapping = {
    "negative": "negative",
    "very negative": "negative",
    "positive": "positive",
    "very positive": "positive",
}
data["label"] = data["sentiment_category"].map(mapping)
data = data.dropna(subset=["label"]).reset_index(drop=True)

print("\nDistribution binaire :")
print(data["label"].value_counts())

sentiment_category
negative         6312
positive         2408
neutral          2146
very negative     679
very positive      65
Name: count, dtype: int64

Distribution binaire :
label
negative    6729
positive    2267
Name: count, dtype: int64


In [12]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))
stop_words.update(["coronavirus", "rt", "mt"])
stop_words -= {"no", "not", "nor", "never", "against"}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)   
    text = re.sub(r"@\w+", "", text)            
    text = re.sub(r"[^a-z\s]", "", text)   
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 1]
    return " ".join(tokens)

data["cleaned_text"] = data["text"].apply(clean_text)
data[["text", "cleaned_text", "label"]].head()

,text,cleaned_text,label
0,Thai doctors treat coronavirus patients succes...,thai doctor treat patient successfully drug co...,positive
1,Saw this on business insider and not on any ot...,saw business insider not mainstream medium rea...,negative
2,New episode the creation of our first hot sauc...,new episode creation first hot sauce check ord...,positive
3,Russia has sent the first plane for evacuation...,russia sent first plane evacuation citizen wuh...,negative
4,MT @osuresearch: The ability to grow the Wuhan...,ability grow wuhan laboratory also make easier...,positive


In [13]:
X_text_train, X_text_test, y_train, y_test = train_test_split(
    data["cleaned_text"], data["label"],
    test_size=0.2, random_state=42, stratify=data["label"]
)

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train = tfidf.fit_transform(X_text_train)
X_test = tfidf.transform(X_text_test)

print("Train :", X_train.shape, "| Test :", X_test.shape)

Train : (7196, 10000) | Test : (1800, 10000)


In [ ]:
models = {
    "SVM": SVC(kernel="linear", class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
for name, model in models.items():
    acc = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro")
    cv_results.append({
        "Modèle": name,
        "CV Accuracy (moy)": acc.mean(),
        "CV Accuracy (std)": acc.std(),
        "CV F1-macro (moy)": f1.mean(),
    })
    print(f"{name:20s} | Accuracy CV : {acc.mean():.4f} (+/- {acc.std():.4f}) | F1-macro : {f1.mean():.4f}")

cv_df = pd.DataFrame(cv_results).sort_values("CV F1-macro (moy)", ascending=False)
cv_df

SVM                  | Accuracy CV : 0.8307 (+/- 0.0097) | F1-macro : 0.7816
Decision Tree        | Accuracy CV : 0.7686 (+/- 0.0049) | F1-macro : 0.7073


In [ ]:
test_results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    test_results.append({
        "Modèle": name,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test F1-macro": f1_score(y_test, y_pred, average="macro"),
    })
    print("=" * 50)
    print(name)
    print(classification_report(y_test, y_pred))

test_df = pd.DataFrame(test_results).sort_values("Test F1-macro", ascending=False)
test_df

In [ ]:
final_df = cv_df.merge(test_df, on="Modèle").sort_values("Test F1-macro", ascending=False)
final_df

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = final_df.melt(id_vars="Modèle",
                        value_vars=["CV Accuracy (moy)", "Test Accuracy"],
                        var_name="Type", value_name="Accuracy")
sns.barplot(data=plot_df, x="Accuracy", y="Modèle", hue="Type", palette="viridis")
plt.title("Comparaison des modèles : validation croisée vs test")
plt.xlim(0.5, 1.0)
plt.show()

In [ ]:
best = final_df.iloc[0]["Modèle"]
cm = confusion_matrix(y_test, predictions[best], labels=["negative", "positive"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["negative", "positive"], yticklabels=["negative", "positive"])
plt.title(f"Matrice de confusion - {best}")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.show()